# Exploratory Data Analysis

**Objective:** Understand the data structure, identify predictive features and document decisions before building the feature engineering pipeline.

**Dataset:** `application_train.csv` : 307 511 loan applications, 122 columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH   = r'C:\Users\Fama\home-credit-risk\data\raw\application_train.csv'
OUTPUT_PATH = r'C:\Users\Fama\home-credit-risk\data\references'

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nDtype breakdown:\n{df.dtypes.value_counts()}')
df.head(3)

In [ ]:
print('TARGET distribution:')
print(df['TARGET'].value_counts())
print(f'\nDefault rate: {df["TARGET"].mean():.2%}')

**Observations:**
- 8.,07% default rate → highly imbalanced dataset
- Modeling will require class weighting (`scale_pos_weight`) or resampling

## Missing Values <a id='3'></a>

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0].reset_index()
missing.columns = ['column', 'missing_rate']

print(f'{len(missing)} columns with missing values')
print(f'\nTop 20:')
display(missing.head(20))

In [ ]:

already_identified = [
    'SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'AMT_CREDIT',
    'AMT_ANNUITY', 'AMT_INCOME_TOTAL', 'AMT_GOODS_PRICE',
    'CODE_GENDER', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'FLAG_MOBIL', 'FLAG_EMAIL', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'
]

# Usable candidated < 20% missing and not yet in the pipeline
candidates = [
    c for c in missing.query('missing_rate < 0.2')['column']
    if c not in already_identified
]

print(f'{len(candidates)} new candidate columns with < 20% missing:')
print('\n'.join(candidates))


- Building-related columns (`COMMONAREA_*`, `FLOORSMIN_*`, `YEARS_BUILD_*`) have 65–70% missing → excluded
- `OWN_CAR_AGE` has 66% missing, expected because since most clients don't own a car
- 15 new candidates with < 20% missing → evaluated below

## Target Analysis <a id='4'></a>

In [ ]:
# Gender gap
print('Default rate by gender:')
print(df.groupby('CODE_GENDER')['TARGET'].agg(['mean', 'count', 'sum']).round(4))

In [ ]:
df['age_group'] = pd.cut(
    -df['DAYS_BIRTH'] / 365,
    bins=[0, 25, 35, 45, 55, 100],
    labels=['<25', '25-35', '35-45', '45-55', '>55']
)

print('Default rate by age group:')
print(df.groupby('age_group', observed=True)['TARGET'].agg(['mean', 'count']).round(3))

- Women: 7.0% default, Men: 10.1% → 3.1 point real gap
- Age follows a clear monotonic pattern: <25 → 12.5%, >55 → 5.3%, younger clients carry significantly higher risk

## Numerical Features <a id='5'></a>

In [ ]:
num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != 'TARGET']

corr_target = (
    df[num_cols].corrwith(df['TARGET'])
    .abs()
    .sort_values(ascending=False)
)

print('Top 20 numerical features by |correlation| with TARGET:')
print(corr_target.head(20).round(4))

In [ ]:
# EXT_SOURCE = normalized scores from external credit bureaus
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    corr = df[col].corr(df['TARGET'])
    nan_rate = df[col].isnull().mean()
    print(f'{col}: corr={corr:.4f}  missing={nan_rate:.2%}')

In [ ]:
# Check redundancy between two region rating columns
r = df['REGION_RATING_CLIENT'].corr(df['REGION_RATING_CLIENT_W_CITY'])
print(f'REGION_RATING_CLIENT vs W_CITY: corr={r:.4f}')
print('→ Keep W_CITY only (more granular, higher corr with TARGET)')

In [ ]:
key_features = [
    'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'REGION_RATING_CLIENT_W_CITY'
]

corr_matrix = df[key_features].corr().round(2)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
plt.xticks(rotation=45, ha='left')
plt.title('Correlation Matrix — Key Features', pad=20, y=-0.08)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}\\correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

- `EXT_SOURCE_2` and `EXT_SOURCE_3` are the strongest predictors (|corr| = 0.16, 0.18) -> must include
- `EXT_SOURCE_1` strong but 56% missing -> include with a binary availability flag
- `AMT_CREDIT` / `AMT_ANNUITY` corr = 0.77 -> keep both, their ratio carries additional signal
- `DAYS_BIRTH` / `DAYS_EMPLOYED` corr = -0.62 -> expected (older = more tenure), both kept
- LightGBM handles residual multicollinearity via tree structure

## Categorical Features <a id='6'></a>

In [ ]:
cat_cols = [c for c in df.select_dtypes('object').columns if c != 'TARGET']

print(f'{"Column":<35} {"Spread":>8} {"N categories":>14}')
print('-' * 60)
for col in cat_cols:
    dr = df.groupby(col)['TARGET'].mean()
    spread = dr.max() - dr.min()
    print(f'{col:<35} {spread:>8.4f} {df[col].nunique():>14}')

In [ ]:
print('Default rate by NAME_INCOME_TYPE:')
print(
    df.groupby('NAME_INCOME_TYPE')['TARGET']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(3)
)

In [ ]:
print('Default rate by OCCUPATION_TYPE:')
print(
    df.groupby('OCCUPATION_TYPE')['TARGET']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(3)
)

In [ ]:
print('Default rate by NAME_EDUCATION_TYPE:')
print(
    df.groupby('NAME_EDUCATION_TYPE')['TARGET']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(3)
)

- `NAME_INCOME_TYPE` spread = 0.40, strongest categorical signal
  - Pensioners (5.4%) vs Working (9.6%), stable income matters
  - Maternity/Unemployed excluded from comparison (< 25 samples each)
- `OCCUPATION_TYPE` spread = 0.123,  Low-skill Laborers at 17.2%, Accountants at 4.8%
- `NAME_EDUCATION_TYPE` clean monotonic relationship: Academic degree (2%) → Lower secondary (11%)
- `WEEKDAY_APPR_PROCESS_START` spread = 0.006 — no predictive value, excluded

## Visualizations <a id='7'></a>

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('Home Credit — EDA Overview', fontsize=14, fontweight='bold')

# Target distribution
axes[0, 0].pie(
    df['TARGET'].value_counts(),
    labels=['No default (92%)', 'Default (8%)'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%', startangle=90
)
axes[0, 0].set_title('Target Distribution')

# Default rate by age group
age_dr = df.groupby('age_group', observed=True)['TARGET'].mean()
axes[0, 1].bar(age_dr.index, age_dr.values,
               color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'])
axes[0, 1].axhline(0.081, color='black', linestyle='--', label='Average 8.1%')
axes[0, 1].set_title('Default Rate by Age Group')
axes[0, 1].set_xlabel('Age Group')
axes[0, 1].set_ylabel('Default Rate')
axes[0, 1].legend()

# EXT_SOURCE_2 density normalized to compensate for class imbalance
df_ext = df[['EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET']].dropna()
for target, color, label in [(0, '#2ecc71', 'No default'), (1, '#e74c3c', 'Default')]:
    axes[1, 0].hist(df_ext[df_ext['TARGET'] == target]['EXT_SOURCE_2'],
                    bins=50, alpha=0.7, color=color, label=label, density=True)
axes[1, 0].set_title('EXT_SOURCE_2 by Target')
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# EXT_SOURCE_3
for target, color, label in [(0, '#f39c12', 'No default'), (1, '#8e44ad', 'Default')]:
    axes[1, 1].hist(df_ext[df_ext['TARGET'] == target]['EXT_SOURCE_3'],
                    bins=50, alpha=0.7, color=color, label=label, density=True)
axes[1, 1].set_title('EXT_SOURCE_3 by Target')
axes[1, 1].set_xlabel('Score')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

# Default rate by income type red/green relative to average
income_dr = df.groupby('NAME_INCOME_TYPE')['TARGET'].mean().sort_values()
income_dr = income_dr[income_dr.index.isin(
    ['Working', 'Commercial associate', 'State servant', 'Pensioner']
)]
axes[2, 0].barh(income_dr.index, income_dr.values,
                color=['#e74c3c' if v > 0.081 else '#2ecc71' for v in income_dr.values])
axes[2, 0].axvline(0.081, color='black', linestyle='--', label='Average 8.1%')
axes[2, 0].set_title('Default Rate by Income Type')
axes[2, 0].set_xlabel('Default Rate')
axes[2, 0].legend()

edu_dr = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values()
axes[2, 1].barh(edu_dr.index, edu_dr.values,
                color=['#e74c3c' if v > 0.081 else '#2ecc71' for v in edu_dr.values])
axes[2, 1].axvline(0.081, color='black', linestyle='--', label='Average 8.1%')
axes[2, 1].set_title('Default Rate by Education Level')
axes[2, 1].set_xlabel('Default Rate')
axes[2, 1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}\\eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Home Credit — Default Rate by Professional Profile', fontsize=14, fontweight='bold')

occ_dr = df.groupby('OCCUPATION_TYPE')['TARGET'].mean().sort_values()
axes[0].barh(occ_dr.index, occ_dr.values,
             color=['#e74c3c' if v > 0.081 else '#2ecc71' for v in occ_dr.values])
axes[0].axvline(0.081, color='black', linestyle='--', label='Average 8.1%')
axes[0].set_title('Default Rate by Occupation')
axes[0].set_xlabel('Default Rate')
axes[0].legend()

edu_dr = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values()
axes[1].barh(edu_dr.index, edu_dr.values,
             color=['#e74c3c' if v > 0.081 else '#2ecc71' for v in edu_dr.values])
axes[1].axvline(0.081, color='black', linestyle='--', label='Average 8.1%')
axes[1].set_title('Default Rate by Education Level')
axes[1].set_xlabel('Default Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}\\eda_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Selection <a id='8'></a>

### New numerical features to add

| Feature | \|corr\| with TARGET | Missing | Rationale |
|---------|----------------------|---------|----------|
| `EXT_SOURCE_2` | 0.161 | 0.2% | External credit score  |
| `EXT_SOURCE_3` | 0.179 | 19.8% | External credit score  |
| `EXT_SOURCE_1` | 0.155 | 56.4% | Strong signal, add with availability flag |
| `REGION_RATING_CLIENT_W_CITY` | 0.061 | 0% | Regional risk score (city-adjusted) |
| `DAYS_ID_PUBLISH` | 0.051 | 0% | Document renewal, stability proxy |
| `REG_CITY_NOT_WORK_CITY` | 0.051 | 0% | Lives and works in different cities |
| `DAYS_REGISTRATION` | 0.042 | 0% | Residential stability |
| `FLAG_DOCUMENT_3` | 0.044 | 0% | Compliance with document requirements |

### New categorical features to add

| Feature | Spread | Rationale |
|---------|--------|-----------|
| `NAME_INCOME_TYPE` | 0.40 | Strongest categorical predictor |
| `OCCUPATION_TYPE` | 0.123 | Clear socio-economic risk gradient |
| `NAME_EDUCATION_TYPE` | 0.091 | Monotonic relationship with default rate |
| `NAME_FAMILY_STATUS` | 0.099 | Family situation affects repayment capacity |

### Excluded features

| Feature | Reason |
|---------|--------|
| `COMMONAREA_*`, `FLOORSMIN_*`, `YEARS_BUILD_*` | 65–70% missing, weak signal |
| `REGION_RATING_CLIENT` | corr = 0.95 with W_CITY version redundant |
| `WEEKDAY_APPR_PROCESS_START` | Spread = 0.006, no predictive value |